# SRQ-FLY Priority 2B — streaming factor quantization
Repeated synthetic Tesla-T4 benchmark. No dataset, feature cache, checkpoint, or held-out split is opened.

In [ ]:
# === Edit path/source values only ===
REPO_GIT_URL='https://github.com/ZaPhat206/SOHO-CL.git'
REPO_BRANCH='experiment/soho-selfcontained'
WORK_DIR='/content/SOHO-CL'
PERSIST_TO_DRIVE=False
DRIVE_OUTPUT='/content/drive/MyDrive/T-SOHO/srq_priority2b_output'
LOCAL_OUTPUT='/content/srq_priority2b_output'

In [ ]:
# Clone/update from a valid parent directory.
import os, subprocess, sys
from pathlib import Path
if PERSIST_TO_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
os.chdir('/content')
repo=Path(WORK_DIR)
if (repo/'.git').is_dir():
    os.chdir(repo)
    subprocess.run(['git','fetch','origin',REPO_BRANCH],check=True)
    subprocess.run(['git','checkout',REPO_BRANCH],check=True)
    subprocess.run(['git','pull','--ff-only','origin',REPO_BRANCH],check=True)
else:
    assert not repo.exists(),f'Non-git path already exists: {repo}'
    subprocess.run(['git','clone','--branch',REPO_BRANCH,'--single-branch',REPO_GIT_URL,WORK_DIR],check=True)
    os.chdir(repo)
subprocess.run([sys.executable,'-m','pip','install','-q','pytest'],check=True)
OUTPUT_DIR=DRIVE_OUTPUT if PERSIST_TO_DRIVE else LOCAL_OUTPUT
print('REPO:',Path.cwd())
print('OUTPUT:',OUTPUT_DIR)
print('COMMIT:',subprocess.check_output(['git','rev-parse','HEAD'],text=True).strip())

In [ ]:
# Immutable source/config identities.
import hashlib, json
def sha(path): return hashlib.sha256(Path(path).read_bytes()).hexdigest()
CONFIG='configs/srq_fly_priority2b_quantization_memory.json'
RUNNER='tools/srq_fly_priority2b_memory_benchmark.py'
EXPECTED={
  CONFIG:'96083a312d4ce15adac5bff9183ada8f8e4ae4b670ac1ebf999626963dd3191f',
  RUNNER:'7ad5a3b640e697e58be7cfcd80fe7ccfeb8c7cfbecd5dde7f1e0d7d2ea7cb094',
  'tools/srq_fly_system_benchmark.py':'ff39a66b9b80f2a0d383f272bbeecce482cd6a1f19b2e41dae1a06b435034a8e',
  'methods/srq_fly_optimized/learner.py':'f4c91e01daf99e55f8de4056d6032da526bf340a615b79dcd84058a07e1863e1',
  'methods/srq_fly_optimized/storage.py':'9d288a3661985da657371e8581f406825d4a8d5e6e0c63381aacda8484490986',
}
for path,expected in EXPECTED.items():
    observed=sha(path); print(path,observed); assert observed==expected,(path,observed,expected)
dirty=subprocess.check_output(['git','status','--porcelain'],text=True).strip()
assert not dirty,f'Repository must be clean before benchmark:\n{dirty}'
assert json.loads(Path(CONFIG).read_text())['seed']==2025
print('PRIORITY-2B IDENTITY GATE: PASS')

In [ ]:
# CPU correctness: byte identity, bounded batches, checkpoints, protocol, resume.
command=[sys.executable,'-m','pytest','-q','tests/test_srq_fly_optimized.py','tests/test_srq_fly_priority2b_memory.py']
completed=subprocess.run(command)
assert completed.returncode==0,'Correctness gate failed; return the full traceback.'
print('PRIORITY-2B CORRECTNESS GATE: PASS')

In [ ]:
# Repeated isolated T4 benchmark. Valid completed workers resume.
assert __import__('torch').cuda.is_available(),'Select a GPU runtime first.'
Path(OUTPUT_DIR).mkdir(parents=True,exist_ok=True)
command=[sys.executable,'-u',RUNNER,'--config',CONFIG,'--output-dir',OUTPUT_DIR,'--device','cuda','--require-clean-git']
print('PRIORITY-2B START: 1 warm-up + 7 measured rounds; 6 workers per round.',flush=True)
print('Wait for START, two TASK lines, and DONE per worker; valid prior workers print RESUME.',flush=True)
completed=subprocess.run(command)
assert completed.returncode==0,'Priority-2B runner failed; return the complete traceback.'
RESULT=Path(OUTPUT_DIR)/'priority2b_memory_results.json'
payload=json.loads(RESULT.read_text())
print('PRIORITY-2B STATUS:',payload['status'])
print('SELECTED:',payload['selected_candidate'])

In [ ]:
# Compact decision table.
import pandas as pd
rows=[]
for item in payload['summaries']:
    rows.append({
      'method':item['label'],
      'median_update_s':item['update_seconds']['median'],
      'median_peak_GiB':item['peak_allocated_bytes']['median']/2**30,
      'persistent_MiB':item['persistent_state_bytes']/2**20,
      'time/eager':item.get('paired_update_ratio_to_eager',{}).get('median'),
      'peak/exact':item.get('paired_peak_allocated_ratio_to_exact',{}).get('median'),
      'quant_peak_delta_MiB':None if item.get('maximum_profiled_quantization_increment_bytes') is None else item['maximum_profiled_quantization_increment_bytes']/2**20,
      'max_logit_drift':item.get('maximum_relative_logit_drift_from_eager'),
      'all_gates':all(item.get('gates',{}).values()) if item.get('gates') else None,
    })
display(pd.DataFrame(rows))
if payload['status']=='STOP_QUANTIZATION_GATE':
    print('STOP: streaming quantization missed a locked gate. Do not relax thresholds.')
else:
    print('PASS: return the ZIP before any dataset experiment.')

In [ ]:
# Show clean absolute and incremental per-stage CUDA peaks.
stage_rows=[]
for item in payload['summaries']:
    profiles=item.get('profiled_task_stage_cuda_memory')
    if not profiles: continue
    for task_index,task in enumerate(profiles,1):
        for stage,memory in task.items():
            stage_rows.append({'method':item['label'],'task':task_index,'stage':stage,'peak_GiB':memory['peak_allocated_bytes']/2**30,'increment_MiB':(memory['peak_allocated_bytes']-memory['before_allocated_bytes'])/2**20})
stage_table=pd.DataFrame(stage_rows)
display(stage_table.sort_values(['method','task','peak_GiB'],ascending=[True,True,False]))

In [ ]:
# Export immutable JSON/probe evidence.
import shutil
bundle=Path('/content/srq_fly_priority2b_memory')
if bundle.exists(): shutil.rmtree(bundle)
shutil.copytree(OUTPUT_DIR,bundle/'output')
shutil.copy2(CONFIG,bundle/'locked_config.json')
shutil.copy2('docs/research/SRQ_FLY_PRIORITY2B_PROTOCOL.md',bundle/'protocol.md')
archive=shutil.make_archive('/content/srq_fly_priority2b_memory','zip',bundle.parent,bundle.name)
print('ZIP:',archive,'bytes=',Path(archive).stat().st_size,'sha256=',sha(archive))
from google.colab import files
files.download(archive)